# 02 — Survival, Segmentation & Product Analytics
**RetainIQ India (Day 5).** Kaplan-Meier retention timing, RFM-style segments, and feature adoption. Each section calls the tested `src/` module so the notebook and the pipeline never diverge.

In [1]:
import sys, pandas as pd
sys.path.insert(0,'../src')
import survival, cohorts, adoption

## Kaplan-Meier retention by contract type

In [2]:
res = survival.run()
pd.DataFrame(res['by_contract']).T

06:01:47 | INFO    | survival | Overall retention: {'S(12)': 0.843, 'S(24)': 0.789, 'S(48)': 0.709, 'S(72)': 0.593, 'median': np.float64(inf)} | median=inf


06:01:47 | INFO    | survival | [Month-to-month] n=3875 S(12)=0.703 S(48)=0.397 median=35.0


06:01:47 | INFO    | survival | [One year] n=1473 S(12)=0.991 S(48)=0.917 median=inf


06:01:47 | INFO    | survival | [Two year] n=1695 S(12)=1.000 S(48)=0.996 median=inf


06:01:47 | INFO    | survival | saved reports/figures/km_by_contract.png


06:01:47 | INFO    | survival | log-rank across contracts: chi2=2352.9  p=0.00e+00


,S(12),S(24),S(48),S(72),median,median_note,n
Month-to-month,0.703,0.586,0.397,0.129,35.0,months,3875
One year,0.991,0.978,0.917,0.568,None,undefined — retention never drops below 50% wi...,1473
Two year,1.0,1.0,0.996,0.936,None,undefined — retention never drops below 50% wi...,1695


Median survival: **month-to-month = 35 months**; one/two-year never drop below 50% retention in-window (median undefined). Log-rank p≈0. Figure: `reports/figures/km_by_contract.png`.

## RFM-style segments (adapted; Recency proxied by tenure — stated honestly)

In [3]:
c = cohorts.run()
pd.DataFrame(c['rfm_segments'])

06:01:47 | INFO    | survival | Tenure-cohort retention:
               customers  churn_rate
tenure_bucket                       
00-12               2186       0.474
13-24               1024       0.287
25-48               1594       0.204
49+                 2239       0.095


06:01:47 | INFO    | survival | RFM-style segments (churn desc):
                        customers  avg_arpu  churn_rate
rfm_segment                                            
New high-value (watch)       1449      85.4       0.583
New / unproven                839      51.9       0.254
Low-engagement               1731      27.8       0.206
Loyal high-value             2072      94.2       0.189
Stable mid                    952      47.8       0.066


06:01:48 | INFO    | survival | Churn by value x loyalty:
loyalty                 New(<=24m)  Tenured(>24m)
customer_value_segment                           
High                         0.626          0.213
Low                          0.196          0.030
Mid                          0.440          0.107


,rfm_segment,customers,avg_arpu,churn_rate
0,New high-value (watch),1449,85.4,0.583
1,New / unproven,839,51.9,0.254
2,Low-engagement,1731,27.8,0.206
3,Loyal high-value,2072,94.2,0.189
4,Stable mid,952,47.8,0.066


Priority segment = **New high-value (watch)**: high ARPU, not yet sticky, ~58% churn.

## Feature adoption — protection add-ons are retention levers

In [4]:
a = adoption.run()
pd.DataFrame(a['per_service'])

06:01:48 | INFO    | survival | Adoption depth vs churn:
 services_held  customers  churn_rate
             1       1264       0.109
             2        859       0.310
             3        846       0.449
             4        965       0.365
             5        922       0.313
             6        908       0.256
             7        676       0.225
             8        395       0.124
             9        208       0.053


06:01:48 | INFO    | survival | Per-service churn (base rate 0.265):


06:01:48 | INFO    | survival | service            churn_with churn_without  ratio


06:01:48 | INFO    | survival | OnlineSecurity          0.146        0.313   0.47


06:01:48 | INFO    | survival | OnlineBackup            0.215        0.292   0.74


...

06:01:48 | INFO    | survival | StreamingTV             0.301        0.243   1.24


06:01:48 | INFO    | survival | StreamingMovies         0.299        0.244   1.23


06:01:48 | INFO    | survival | MultipleLines           0.286         0.25   1.14


06:01:48 | INFO    | survival | saved reports/figures/adoption_depth_churn.png


,service,churn_with,churn_without,ratio
0,OnlineSecurity,0.146,0.313,0.47
1,OnlineBackup,0.215,0.292,0.74
2,DeviceProtection,0.225,0.287,0.78
3,TechSupport,0.152,0.312,0.49
4,StreamingTV,0.301,0.243,1.24
5,StreamingMovies,0.299,0.244,1.23
6,MultipleLines,0.286,0.250,1.14


OnlineSecurity/TechSupport roughly **halve** churn (ratio ~0.47–0.49); streaming slightly raises it. Offer design → protection bundles.